# Predicting enhancer–gene regulation from random walks + selective SSMs

**CS224W project.** Can we predict whether an enhancer regulates a gene from random walks over the gene's
regulatory neighborhood (a linear-genome chain plus Hi-C contact edges), encoded by a bidirectional Mamba?

This notebook is organized as the full project pipeline:

| Part | What it does | Status |
|---|---|---|
| **1. Data** | Download and harmonize the CRISPR benchmark, K562 ATAC-seq, K562 Hi-C, and GRCh38 | **this section** |
| 2. Graph | Build the per-chromosome chain + Hi-C graph | next |
| 3. Walks | node2vec-style second-order walks with $(p, q, \beta)$ | |
| 4. Model | Walk tokenization, bidirectional Mamba encoder, pair MLP | |
| 5. Evaluation | Chromosome-held-out CV, AUPRC, precision at 70% recall vs. ABC / ENCODE-rE2G | |

## Part 1 overview: getting the data into a workable format

Everything in the model is defined on a single set of genomic intervals, the **candidate elements**
(ABC's definition: strong ATAC peaks extended to 500 bp, plus every gene promoter). Part 1 builds that
node set and attaches every data source to it:

| Source | Raw file | Becomes |
|---|---|---|
| ENCODE-rE2G K562 CRISPR benchmark (Gschwind et al.) | `EPCrisprBenchmark_combined_data.training_K562.GRCh38.tsv.gz` | labeled (element, gene) pairs |
| ABC reference gene list (RefSeq, collapsed) | `CollapsedGeneBounds.hg38.bed` | promoter nodes (TSS $\pm$ 250 bp) |
| ENCODE K562 ATAC-seq, ENCSR483RKN | peaks `ENCFF558BLC`, BAMs `ENCFF512VEZ` + `ENCFF987XOV` | candidate elements + activity feature $a_v$ |
| ENCODE K562 intact Hi-C, ENCSR479XDG | `ENCFF621AIY.hic` (34 GB, **streamed**, not downloaded) | observed and O/E contacts between node bins |
| UCSC hg38 | `hg38.fa` | node sequences (for HyenaDNA embeddings $s_v$) |

These choices match the ENCODE-rE2G / ABC pipelines (same Hi-C file, same reference gene list, same
blocklist, same candidate-element recipe), so our baselines are comparable.

**Outputs** (all in `data/processed/`):

- `crispr_pairs.parquet`: benchmark pairs with node mappings
- `nodes.parquet`: candidate elements with activity, promoter annotations, Hi-C bin
- `hic/{chrom}.parquet`: Hi-C contacts between bins that contain nodes, within 5 Mb
- `node_sequences.parquet`: DNA sequence per node

## 1.0 Setup

The environment is pinned in `environment.yml` (works on macOS for development and on Linux clusters such as
Sherlock and Chimera). On Colab, the next cell pip-installs the same packages instead.

Paths default to `./data`. On a cluster, point `E2G_DATA_DIR` at scratch space (e.g.
`export E2G_DATA_DIR=$SCRATCH/e2g_data`) before launching Jupyter, so the ~10 GB of raw data doesn't land in
your home directory.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Colab ships torch / pandas / sklearn; add the genomics I/O packages.
    %pip install -q pysam pyBigWig pyfaidx "pyranges<0.2" hic-straw torch_geometric

In [ ]:
import gzip, shutil, time, warnings
from concurrent.futures import as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import pyranges as pr
import pysam
import pyfaidx
import hicstraw
from sklearn.metrics import average_precision_score, precision_recall_curve
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)

# ---- paths ---------------------------------------------------------------
DATA_DIR = Path(os.environ.get("E2G_DATA_DIR", "data")).resolve()
RAW = DATA_DIR / "raw"
PROC = DATA_DIR / "processed"
HIC_DIR = PROC / "hic"
FIG_DIR = Path("figures")
for d in (RAW, PROC, HIC_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- data config (mirrors ABC / ENCODE-rE2G defaults) --------------------
CHROMS = [f"chr{i}" for i in range(1, 23)] + ["chrX"]   # benchmark has no chrY pairs
PEAK_EXTEND = 250          # candidate element = summit +/- 250 bp
N_STRONGEST = 150_000      # keep the 150k strongest ATAC windows
PROMOTER_HALF = 250        # promoter = TSS +/- 250 bp
HIC_RES = 5_000            # Hi-C bin size (bp)
HIC_NORM = "SCALE"         # Juicer SCALE normalization (the .hic's default for intact Hi-C)
MAX_DIST = 5_000_000       # only keep contacts within 5 Mb (ABC's neighborhood)

print("data dir:", DATA_DIR)

In [ ]:
# Plot style: recessive axes, thin marks, a fixed categorical order.
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e4e3df", "grid.linewidth": 0.6,
    "axes.edgecolor": "#8d8c87", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "lines.linewidth": 2,
    "legend.frameon": False, "font.size": 10,
})
C_NEG, C_POS, C_3 = "#2a78d6", "#eb6834", "#1baf7a"     # categorical slots 1-3
DATASET_COLORS = {"Gasperini2019": C_NEG, "Nasser2021": C_POS, "Schraivogel2020": C_3}

## 1.1 Download raw data

Everything is public. Small files (benchmark, references, peaks) download in seconds. The two large
downloads are:

- **K562 ATAC-seq BAMs** (2 replicates, ~6 GB total). We need actual read counts in every candidate
  element (ABC's activity term and our node feature $a_v$), and ENCODE does not host BAM indexes, so we
  download and index them locally. Set `DOWNLOAD_BAMS = False` to skip if they already exist elsewhere.
- **hg38 FASTA** (~1 GB gzipped, 3.1 GB uncompressed). Needed for node sequences / HyenaDNA. `pyfaidx`
  needs an uncompressed (or bgzipped) FASTA for random access, so we decompress it once.

The **Hi-C file is not downloaded.** `ENCFF621AIY.hic` is 34 GB, but the `.hic` format is indexed, and
`hicstraw` fetches only the byte ranges it needs over HTTP (Section 1.6).

Downloads are idempotent: a file that already exists is skipped, and interrupted downloads resume.

In [ ]:
ENCODE = "https://www.encodeproject.org/files/{acc}/@@download/{acc}.{ext}"
ABC_REF = "https://raw.githubusercontent.com/broadinstitute/ABC-Enhancer-Gene-Prediction/main/reference"
CRISPR_REPO = "https://raw.githubusercontent.com/EngreitzLab/CRISPR_comparison/main/resources/crispr_data"

FILES = {
    # CRISPR ground truth (Gschwind et al. 2023, ENCODE-rE2G training set)
    "crispr": (f"{CRISPR_REPO}/EPCrisprBenchmark_combined_data.training_K562.GRCh38.tsv.gz",
               "EPCrisprBenchmark_combined_data.training_K562.GRCh38.tsv.gz"),
    # ABC reference files (hg38)
    "genes": (f"{ABC_REF}/hg38/CollapsedGeneBounds.hg38.bed", "CollapsedGeneBounds.hg38.bed"),
    "chrom_sizes": (f"{ABC_REF}/hg38/GRCh38_EBV.no_alt.chrom.sizes.tsv", "GRCh38_EBV.no_alt.chrom.sizes.tsv"),
    "blocklist": (f"{ABC_REF}/hg38/GRCh38_unified_blacklist.bed", "GRCh38_unified_blacklist.bed"),
    "ubiquitous": (f"{ABC_REF}/UbiquitouslyExpressedGenes.txt", "UbiquitouslyExpressedGenes.txt"),
    # K562 ATAC-seq (ENCSR483RKN): pooled pseudoreplicated MACS2 peaks
    "atac_peaks": (ENCODE.format(acc="ENCFF558BLC", ext="bed.gz"), "ENCFF558BLC.bed.gz"),
}
LARGE_FILES = {
    "atac_bam_rep1": (ENCODE.format(acc="ENCFF512VEZ", ext="bam"), "ENCFF512VEZ.bam"),
    "atac_bam_rep2": (ENCODE.format(acc="ENCFF987XOV", ext="bam"), "ENCFF987XOV.bam"),
    "hg38_gz": ("https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz", "hg38.fa.gz"),
}
HIC_URL = ENCODE.format(acc="ENCFF621AIY", ext="hic")   # K562 intact Hi-C (ENCSR479XDG), streamed
DOWNLOAD_BAMS = True


def download(url, dest, chunk=1 << 20):
    # Stream url -> dest, resuming a partial dest.part if present. Skips if dest exists.
    dest = Path(dest)
    if dest.exists():
        return dest
    part = dest.with_name(dest.name + ".part")
    pos = part.stat().st_size if part.exists() else 0
    with requests.get(url, stream=True, timeout=60,
                      headers={"Range": f"bytes={pos}-"} if pos else {}) as r:
        r.raise_for_status()
        resumed = r.status_code == 206
        total = int(r.headers.get("content-length", 0)) + (pos if resumed else 0)
        with open(part, "ab" if resumed else "wb") as f, \
             tqdm(total=total, initial=pos if resumed else 0, unit="B", unit_scale=True,
                  desc=dest.name, leave=False) as bar:
            for block in r.iter_content(chunk):
                f.write(block)
                bar.update(len(block))
    part.rename(dest)
    return dest


PATHS = {k: download(url, RAW / name) for k, (url, name) in FILES.items()}
for k, (url, name) in LARGE_FILES.items():
    if k.startswith("atac_bam") and not DOWNLOAD_BAMS:
        PATHS[k] = RAW / name
        continue
    PATHS[k] = download(url, RAW / name)

# Decompress hg38 once for random access.
PATHS["hg38"] = RAW / "hg38.fa"
if not PATHS["hg38"].exists():
    with gzip.open(PATHS["hg38_gz"], "rb") as src, open(PATHS["hg38"], "wb") as dst:
        shutil.copyfileobj(src, dst, length=16 << 20)

pd.DataFrame([(k, p.name, f"{p.stat().st_size / 1e6:,.1f} MB" if p.exists() else "missing")
              for k, p in PATHS.items()], columns=["key", "file", "size"])

## 1.2 The CRISPR benchmark

Each row is one tested (element, gene) pair from a CRISPRi screen in K562. The columns we use:

- `chrom, chromStart, chromEnd`: the perturbed element
- `chrTSS, startTSS`: the measured gene's TSS (a 1 bp interval)
- `measuredGeneSymbol`: gene
- `Regulated`: **the label**, `True` iff knocking down the element significantly *decreased* expression
  (adjusted $p < 0.05$ and negative effect size)
- `Dataset`: which screen (Gasperini 2019, Nasser 2021, Schraivogel 2020)
- `distanceToTSS`: element center to TSS

The file also carries ENCODE chromatin annotations (H3K27ac, DNase RPM, ...) and power estimates.
We keep those for analysis but **do not use them as model inputs**; the model sees only ATAC, Hi-C,
and sequence.

First we check that the counts match the proposal's Table 1.

In [ ]:
crispr = pd.read_csv(PATHS["crispr"], sep="\t")
crispr["label"] = crispr["Regulated"].astype(int)
crispr["elem_center"] = (crispr["chromStart"] + crispr["chromEnd"]) // 2
crispr["elem_len"] = crispr["chromEnd"] - crispr["chromStart"]
crispr["pair_id"] = np.arange(len(crispr))

assert (crispr["chrom"] == crispr["chrTSS"]).all(), "all pairs are cis"
assert crispr["ValidConnection"].all()

summary = crispr.groupby("Dataset")["label"].agg(pairs="size", positives="sum")
summary.loc["Total"] = summary.sum()
summary["positive rate"] = (summary["positives"] / summary["pairs"]).map("{:.1%}".format)
print(f"{crispr.shape[0]:,} pairs | {crispr[['chrom','chromStart','chromEnd']].drop_duplicates().shape[0]:,} "
      f"unique elements | {crispr['measuredGeneSymbol'].nunique():,} unique genes")
summary

Matches the proposal: 10,356 pairs and 471 positives (4.5%). There are far fewer unique elements and genes
than pairs, because each screen tests every element against every gene in a window around it.

Two properties matter for the model and for evaluation:

1. **Elements vary in length.** Most are about 500 bp (they were defined from DHS peaks), but some are
   tile-scan fragments shorter than 100 bp and some are merged regions up to 4 kb. We map each element onto
   our candidate nodes by overlap (Section 1.5), not by exact coordinates.
2. **Chromosomes are unevenly covered** (chr19 and chr11 alone hold about 35% of pairs, from the
   Gasperini and Fulco loci), so chromosome-held-out folds need to be balanced by pair count, not by
   chromosome count.

In [ ]:
per_chrom = (crispr.groupby("chrom")["label"].agg(pairs="size", positives="sum")
             .reindex([c for c in CHROMS if c in crispr["chrom"].unique()]))
per_chrom["positive rate"] = per_chrom["positives"] / per_chrom["pairs"]
per_chrom.sort_values("pairs", ascending=False).T.round(3)

### 1.2.1 Summary statistics: enhancer–gene distances

The genomic distance between an element and its target TSS is the strongest single feature for this task
(rE2G's distance-only baseline is competitive with ABC), so it is the first thing to look at. It also sets
the scale our walks must cover: if positives sit at about 10–50 kb but negatives extend to over 1 Mb,
walks only need to reach a few hundred nodes along the chain.

In [ ]:
crispr["dist_kb"] = crispr["distanceToTSS"].abs() / 1e3

def dist_summary(g):
    d = g["dist_kb"]
    return pd.Series({
        "n": len(d), "median kb": d.median(), "mean kb": d.mean(),
        "IQR kb": f"{d.quantile(.25):.1f}–{d.quantile(.75):.1f}",
        "90th pct kb": d.quantile(.9), "max kb": d.max(),
        "< 10 kb": (d < 10).mean(), "< 100 kb": (d < 100).mean(), "< 1 Mb": (d < 1000).mean(),
    })

dist_tab = (crispr.assign(Label=crispr["Regulated"].map({True: "positive", False: "negative"}))
            .groupby(["Dataset", "Label"]).apply(dist_summary, include_groups=False))
overall = (crispr.assign(Label=crispr["Regulated"].map({True: "positive", False: "negative"}))
           .groupby("Label").apply(dist_summary, include_groups=False))
overall.index = pd.MultiIndex.from_product([["All"], overall.index])
dist_tab = pd.concat([dist_tab, overall])
dist_tab.style.format({"median kb": "{:.1f}", "mean kb": "{:.1f}", "90th pct kb": "{:.0f}", "max kb": "{:.0f}",
                       "< 10 kb": "{:.0%}", "< 100 kb": "{:.0%}", "< 1 Mb": "{:.0%}", "n": "{:.0f}"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

# (a) distance distribution by label (log-scaled x; densities so the 4.5% class is visible)
bins = np.logspace(-1, np.log10(crispr["dist_kb"].max() * 1.05), 45)
ax = axes[0]
for lab, color, name in [(0, C_NEG, "negative"), (1, C_POS, "positive")]:
    d = crispr.loc[crispr["label"] == lab, "dist_kb"]
    ax.hist(d, bins=bins, density=True, histtype="step", color=color, lw=2,
            label=f"{name} (n={len(d):,}, median {d.median():.0f} kb)")
ax.set_xscale("log"); ax.set_xlabel("|element center − TSS| (kb)"); ax.set_ylabel("density")
ax.set_title("Enhancer–TSS distance by label", loc="left"); ax.legend(loc="upper left", fontsize=8)

# (b) positive rate vs. distance, per screen
edges_kb = np.array([0, 5, 10, 25, 50, 100, 250, 500, 1000, 5000])
crispr["dist_bin"] = pd.cut(crispr["dist_kb"], edges_kb, right=False)
rate = crispr.groupby(["Dataset", "dist_bin"], observed=True)["label"].agg(["mean", "size"]).reset_index()
centers = {iv: np.sqrt(max(iv.left, 1) * iv.right) for iv in crispr["dist_bin"].cat.categories}
ax = axes[1]
for ds, g in rate.groupby("Dataset"):
    g = g[g["size"] >= 20]  # hide bins too small to estimate a rate
    ax.plot([centers[b] for b in g["dist_bin"]], g["mean"], marker="o", ms=5,
            color=DATASET_COLORS[ds], label=ds)
ax.set_xscale("log"); ax.set_xlabel("distance bin center (kb)"); ax.set_ylabel("fraction regulated")
ax.set_title("Positive rate falls off with distance", loc="left"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "crispr_distance.png", bbox_inches="tight"); plt.show()

crispr.groupby("dist_bin", observed=True)["label"].agg(pairs="size", positives="sum", rate="mean").T.round(3)

**Distance-only baseline.** Scoring each pair by $1/d$ gives the floor every model must beat. AUPRC
depends on the base rate, so we also report the random baseline (equal to the positive rate) for each screen.

In [ ]:
def precision_at_recall(y, s, recall=0.7):
    prec, rec, _ = precision_recall_curve(y, s)
    return prec[rec >= recall].max()

rows = []
for ds, g in list(crispr.groupby("Dataset")) + [("Pooled", crispr)]:
    s = -g["dist_kb"]
    rows.append({"Dataset": ds, "pairs": len(g), "random AUPRC": g["label"].mean(),
                 "distance AUPRC": average_precision_score(g["label"], s),
                 "distance precision @70% recall": precision_at_recall(g["label"], s)})
baseline_distance = pd.DataFrame(rows).set_index("Dataset")
baseline_distance.round(3)

## 1.3 Gene promoters

Genes enter the graph as **promoter nodes** (TSS $\pm$ 250 bp). We use ABC's reference gene list (RefSeq
curated, collapsed to one TSS per gene). However, the benchmark's TSS for a gene sometimes differs from ABC's,
by up to 10 kb when an alternative TSS is used. A benchmark gene must map to the promoter node at the
benchmark's own TSS, so **for benchmark genes we override the ABC TSS with the benchmark TSS.**

In [ ]:
genes = pd.read_csv(PATHS["genes"], sep="\t").rename(columns={"#chr": "chrom", "name": "gene"})
genes = genes[genes["chrom"].isin(CHROMS)].copy()
genes["tss"] = np.where(genes["strand"] == "+", genes["start"], genes["end"] - 1)

bench_tss = crispr.groupby("measuredGeneSymbol").agg(chrTSS=("chrTSS", "first"), startTSS=("startTSS", "first"),
                                                     n_tss=("startTSS", "nunique"))
assert (bench_tss["n_tss"] == 1).all(), "one TSS per benchmark gene"
missing = set(bench_tss.index) - set(genes["gene"])
print(f"benchmark genes missing from ABC list: {len(missing)}")

genes = genes.merge(bench_tss[["startTSS"]], left_on="gene", right_index=True, how="left")
genes["tss_source"] = np.where(genes["startTSS"].notna(), "benchmark", "ABC")
shift = (genes["startTSS"] - genes["tss"]).dropna().abs()
print(f"benchmark vs ABC TSS: identical for {(shift == 0).mean():.0%}, within 250 bp for {(shift <= 250).mean():.0%}, "
      f"max shift {shift.max() / 1e3:,.0f} kb")
genes["tss"] = genes["startTSS"].fillna(genes["tss"]).astype(int)
genes["is_benchmark_gene"] = genes["tss_source"] == "benchmark"

promoters = genes.assign(Start=(genes["tss"] - PROMOTER_HALF).clip(lower=0),
                         End=genes["tss"] + PROMOTER_HALF)[["chrom", "Start", "End", "gene", "strand", "gene_type",
                                                             "is_benchmark_gene"]]
print(f"{len(promoters):,} promoters; gene types:", promoters["gene_type"].value_counts().head(4).to_dict())

## 1.4 Candidate elements = graph nodes (ABC recipe)

From the proposal: *call ATAC peaks with MACS2, center a 500 bp window on each, keep the 150,000 windows with
highest read coverage, merge overlaps, add every gene's promoter (TSS $\pm$ 250 bp).* Concretely:

1. **Peaks.** ENCODE's pooled pseudoreplicated MACS2 peaks for K562 ATAC (`ENCFF558BLC`). One peak region can
   list several summits, so we deduplicate summits.
2. **Windows.** summit $\pm$ 250 bp; drop windows that hit the ENCODE blocklist.
3. **Strength.** Count ATAC reads (both replicates) in each window and keep the top 150,000.
4. **Merge** overlapping windows, **union with promoters**, merge again, then remove blocklist overlaps.
5. **Activity.** Recount reads in the final elements and normalize to RPM:

$$
a_v = \log\left(1 + \mathrm{RPM}_v\right)
$$

One small deviation from ABC: ABC ranks peaks by reads in the variable-width MACS2 peak, while we rank fixed
500 bp windows. Equal widths make the counts directly comparable.

In [ ]:
def to_pr(df, chrom="chrom", start="start", end="end"):
    return pr.PyRanges(df.rename(columns={chrom: "Chromosome", start: "Start", end: "End"}))

def from_pr(g):
    return g.df.rename(columns={"Chromosome": "chrom", "Start": "start", "End": "end"})

NARROWPEAK = ["chrom", "start", "end", "name", "score", "strand", "signal", "p", "q", "summit"]
peaks = pd.read_csv(PATHS["atac_peaks"], sep="\t", header=None, names=NARROWPEAK)
peaks = peaks[peaks["chrom"].isin(CHROMS)]
peaks["summit_pos"] = peaks["start"] + peaks["summit"]
summits = peaks.drop_duplicates(["chrom", "summit_pos"])
print(f"{len(peaks):,} peak records -> {len(summits):,} unique summits")

windows = pd.DataFrame({"chrom": summits["chrom"].values,
                        "start": (summits["summit_pos"] - PEAK_EXTEND).clip(lower=0).values,
                        "end": (summits["summit_pos"] + PEAK_EXTEND).values})
blocklist = pd.read_csv(PATHS["blocklist"], sep="\t", header=None, usecols=[0, 1, 2], names=["chrom", "start", "end"])
blocklist_pr = to_pr(blocklist)
windows = from_pr(to_pr(windows).overlap(blocklist_pr, invert=True))
print(f"{len(windows):,} windows after blocklist filter")

**Read counting.** We index each BAM once (`.bai`, about a minute each), then use `pysam` to count reads
overlapping each interval. ENCODE's "alignments" BAMs are already filtered (MAPQ, duplicates, chrM removed).
RPM uses the total mapped reads on the chromosomes we model.

In [ ]:
BAMS = [PATHS["atac_bam_rep1"], PATHS["atac_bam_rep2"]]
for b in BAMS:
    if not Path(str(b) + ".bai").exists():
        print("indexing", b.name); pysam.index(str(b))

def mapped_reads(bam_path):
    with pysam.AlignmentFile(bam_path) as bam:
        return sum(s.mapped for s in bam.get_index_statistics() if s.contig in CHROMS)

def count_reads(df, bam_paths=BAMS):
    # Reads (summed over replicates) overlapping each [start, end) interval in df.
    counts = np.zeros(len(df), dtype=np.int64)
    for p in bam_paths:
        with pysam.AlignmentFile(p) as bam:
            for i, (c, s, e) in enumerate(tqdm(zip(df["chrom"], df["start"], df["end"]), total=len(df),
                                              desc=Path(p).name, leave=False, mininterval=2)):
                counts[i] += bam.count(c, int(s), int(e))
    return counts

TOTAL_READS = sum(mapped_reads(b) for b in BAMS)
print(f"total mapped reads (both replicates, {len(CHROMS)} chromosomes): {TOTAL_READS:,}")

windows["reads"] = count_reads(windows)
strong = windows.nlargest(N_STRONGEST, "reads")
print(f"top {N_STRONGEST:,} windows: min reads {strong['reads'].min()}, median {strong['reads'].median():.0f}")

In [ ]:
merged_peaks = to_pr(strong[["chrom", "start", "end"]]).merge()
prom_pr = to_pr(promoters[["chrom", "Start", "End"]], start="Start", end="End")
nodes_pr = pr.concat([merged_peaks, prom_pr]).merge().overlap(blocklist_pr, invert=True)

nodes = from_pr(nodes_pr)
nodes["chrom"] = pd.Categorical(nodes["chrom"], categories=CHROMS, ordered=True)
nodes = nodes.sort_values(["chrom", "start"]).reset_index(drop=True)
nodes["chrom"] = nodes["chrom"].astype(str)
nodes["node_id"] = np.arange(len(nodes))
nodes["center"] = (nodes["start"] + nodes["end"]) // 2
nodes["length"] = nodes["end"] - nodes["start"]

# Promoter annotation: which genes' promoters each node contains.
pj = from_pr(to_pr(nodes[["chrom", "start", "end", "node_id"]]).join(
    to_pr(promoters, start="Start", end="End"), suffix="_p"))
prom_genes = pj.groupby("node_id")["gene"].agg(lambda g: ",".join(sorted(set(g))))
nodes["promoter_genes"] = nodes["node_id"].map(prom_genes).fillna("")
nodes["is_promoter"] = nodes["promoter_genes"] != ""
nodes["has_peak"] = nodes["node_id"].isin(
    from_pr(to_pr(nodes[["chrom", "start", "end", "node_id"]]).overlap(merged_peaks))["node_id"])

# Activity on the final intervals.
nodes["atac_reads"] = count_reads(nodes)
nodes["atac_rpm"] = nodes["atac_reads"] / TOTAL_READS * 1e6
nodes["log_atac"] = np.log1p(nodes["atac_rpm"])
nodes["hic_bin"] = nodes["center"] // HIC_RES * HIC_RES

print(f"{len(nodes):,} candidate elements | promoter nodes: {nodes['is_promoter'].sum():,} "
      f"| ATAC-only: {(~nodes['is_promoter']).sum():,} | promoter nodes with ATAC peak: "
      f"{(nodes['is_promoter'] & nodes['has_peak']).sum():,}")
nodes.groupby("is_promoter")[["length", "atac_reads", "log_atac"]].describe().T.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
ax = axes[0]
ax.hist(nodes["length"].clip(upper=3000), bins=60, color=C_NEG)
ax.set_yscale("log"); ax.set_xlabel("element length (bp, clipped at 3 kb)"); ax.set_ylabel("count")
ax.set_title("Node lengths", loc="left")

ax = axes[1]
b = np.linspace(0, nodes["log_atac"].quantile(.999), 60)
for flag, color, name in [(False, C_NEG, "ATAC peak"), (True, C_POS, "promoter")]:
    ax.hist(nodes.loc[nodes["is_promoter"] == flag, "log_atac"], bins=b, histtype="step", lw=2, color=color,
            label=name, density=True)
ax.set_xlabel(r"$a_v = \log(1 + \mathrm{RPM})$"); ax.set_ylabel("density"); ax.legend()
ax.set_title("Activity feature", loc="left")

ax = axes[2]
gaps = nodes.groupby("chrom")["center"].diff().dropna()
ax.hist(np.log10(gaps.clip(lower=1)), bins=60, color=C_NEG)
ax.set_xlabel(r"$\log_{10}$ gap to next node (bp)"); ax.set_ylabel("count")
ax.set_title(f"Chain step size (median {gaps.median() / 1e3:.1f} kb)", loc="left")
fig.tight_layout(); fig.savefig(FIG_DIR / "nodes.png", bbox_inches="tight"); plt.show()

The rightmost panel matters for walk design: the median linear step between adjacent nodes is the
"unit" of a chain step. A walk of $L$ linear steps covers about $L$ times that median gap, which tells us how
long walks need to be to reach the distances in Section 1.2.1.

## 1.5 Map benchmark pairs onto nodes

- **Gene → node**: the node containing the gene's (benchmark) TSS. Since benchmark promoters were added to the
  node set, this succeeds unless the TSS falls in the blocklist.
- **Element → node**: the node with the largest overlap with the CRISPR element. Some tested elements have no
  ATAC signal and overlap no candidate node. ENCODE-rE2G's benchmarking (CRISPR_comparison) handles this by
  scoring such pairs as 0, i.e. they count as predicted negatives and still enter AUPRC. We keep them
  (`elem_node = -1`) and follow the same convention at evaluation time, so our numbers are comparable.

In [ ]:
node_pr = to_pr(nodes[["chrom", "start", "end", "node_id"]])

# element -> best-overlapping node
elems = crispr[["chrom", "chromStart", "chromEnd"]].drop_duplicates().reset_index(drop=True)
elems["elem_key"] = np.arange(len(elems))
ej = to_pr(elems, start="chromStart", end="chromEnd").join(node_pr, report_overlap=True, suffix="_n").df
best = ej.sort_values("Overlap", ascending=False).drop_duplicates("elem_key")
elems = elems.merge(best[["elem_key", "node_id", "Overlap"]], on="elem_key", how="left")
elems = elems.merge(ej.groupby("elem_key").size().rename("n_elem_nodes"), on="elem_key", how="left")
elems = elems.rename(columns={"node_id": "elem_node", "Overlap": "elem_overlap_bp"})

# gene TSS -> node
tss = crispr[["chrTSS", "startTSS"]].drop_duplicates().assign(endTSS=lambda d: d["startTSS"] + 1)
tj = to_pr(tss, chrom="chrTSS", start="startTSS", end="endTSS").join(node_pr, suffix="_n").df
tss = tss.merge(tj.rename(columns={"Chromosome": "chrTSS", "Start": "startTSS"})[["chrTSS", "startTSS", "node_id"]],
                on=["chrTSS", "startTSS"], how="left").rename(columns={"node_id": "gene_node"})

pairs = (crispr.merge(elems, on=["chrom", "chromStart", "chromEnd"], how="left")
               .merge(tss[["chrTSS", "startTSS", "gene_node"]], on=["chrTSS", "startTSS"], how="left"))
pairs[["elem_node", "gene_node", "n_elem_nodes"]] = pairs[["elem_node", "gene_node", "n_elem_nodes"]].fillna(-1).astype(int)
pairs["elem_overlap_bp"] = pairs["elem_overlap_bp"].fillna(0).astype(int)
pairs["elem_in_graph"] = pairs["elem_node"] >= 0
pairs["elem_is_gene_node"] = pairs["elem_node"] == pairs["gene_node"]
assert len(pairs) == len(crispr)

print(f"genes mapped to a node: {(pairs['gene_node'] >= 0).mean():.2%}")
cov = pairs.groupby("Dataset").agg(pairs=("label", "size"), elem_in_graph=("elem_in_graph", "mean"),
                                   positives=("label", "sum"))
cov["positives_in_graph"] = pairs[pairs["label"] == 1].groupby("Dataset")["elem_in_graph"].mean()
cov.loc["Pooled"] = [len(pairs), pairs["elem_in_graph"].mean(), pairs["label"].sum(),
                     pairs.loc[pairs["label"] == 1, "elem_in_graph"].mean()]
print(f"elements spanning >1 node: {(pairs.drop_duplicates('elem_key')['n_elem_nodes'] > 1).mean():.1%}; "
      f"element node == gene node: {pairs['elem_is_gene_node'].sum()} pairs")
cov.style.format({"elem_in_graph": "{:.1%}", "positives_in_graph": "{:.1%}", "pairs": "{:.0f}", "positives": "{:.0f}"})

Positives are covered better than negatives, as expected: regulatory elements are accessible. Pairs whose element
is outside the graph are a recall ceiling shared by every ATAC-based method (ABC and ENCODE-rE2G included).

## 1.6 Hi-C contacts (streamed from ENCODE)

We need two quantities for each pair of nodes within 5 Mb:

- $O_{uv}$: SCALE-normalized observed contacts between the 5 kb bins containing $u$ and $v$ (ABC's contact term)
- $O_{uv}/E_{uv}$: observed over expected at that distance (our Hi-C edge weight $\pi_{uv}$ and edge threshold $\tau$)

A full 5 kb matrix within a 5 Mb band is about $10^8$ entries genome-wide. We only need bins that contain a node
(or a benchmark element/TSS), which cuts this by more than an order of magnitude. For each chromosome we stream
5 Mb blocks of rows, each paired with columns out to +5 Mb, via `hicstraw` (byte-range HTTP requests into the
34 GB file), keep only the relevant bins, and cache the result to `data/processed/hic/{chrom}.parquet`.

**O/E without a second fetch.** `hicstraw`'s `"oe"` matrix is exactly observed divided by the file's expected
vector at that diagonal offset, $E_{uv} = \bar{E}\left(|b_u - b_v| / 5\mathrm{kb}\right)$ (checked against straw's own
output). So we fetch only the observed matrix and divide by the expected vector, halving the network traffic.

**Runtime.** Each 5 Mb $\times$ 10 Mb block takes about 3–5 s over HTTP, and the genome has about 620 blocks. `hicstraw`
holds the GIL (threads give no speedup), so chromosomes are fetched in parallel **processes** (`HIC_WORKERS`,
using `fork` so notebook-defined functions carry over). This runs once, and cached chromosomes are skipped. On a
cluster with the 34 GB file downloaded locally, set `HIC_URL` to the local path for a large speedup.

In [ ]:
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor

HIC_BLOCK = 5_000_000
HIC_WORKERS = 8

hic = hicstraw.HiCFile(HIC_URL)
CHROM_LEN = {c.name: c.length for c in hic.getChromosomes()}
print("Hi-C genome:", hic.getGenomeID(), "| resolutions:", hic.getResolutions())

# Bins we care about: node bins plus benchmark element / TSS bins (so every benchmark pair has a contact value).
bins_of_interest = {c: set(g) for c, g in nodes.groupby("chrom")["hic_bin"]}
for c, g in pairs.groupby("chrom"):
    bins_of_interest[c] |= set(g["elem_center"] // HIC_RES * HIC_RES) | set(g["startTSS"] // HIC_RES * HIC_RES)

def _records_df(records):
    return pd.DataFrame({"bin1": [r.binX for r in records], "bin2": [r.binY for r in records],
                         "v": [r.counts for r in records]})

def fetch_hic_chrom(chrom, url=HIC_URL):
    # Banded intra-chromosomal contacts (observed + O/E) between bins of interest, within MAX_DIST.
    keep = np.array(sorted(bins_of_interest[chrom]))
    f = hicstraw.HiCFile(url)  # one handle per process
    obs = f.getMatrixZoomData(chrom, chrom, "observed", HIC_NORM, "BP", HIC_RES)
    oe = f.getMatrixZoomData(chrom, chrom, "oe", HIC_NORM, "BP", HIC_RES)
    oe.getRecords(0, HIC_RES, 0, HIC_RES)            # a tiny fetch loads the expected vector
    expected = np.asarray(oe.getExpectedValues())
    L, out = CHROM_LEN[chrom], []
    for s in range(0, L, HIC_BLOCK):
        x1, x2 = s, min(s + HIC_BLOCK, L) - 1
        y2 = min(s + HIC_BLOCK + MAX_DIST, L) - 1
        d = _records_df(obs.getRecords(x1, x2, x1, y2))
        lo, hi = np.minimum(d["bin1"], d["bin2"]), np.maximum(d["bin1"], d["bin2"])
        d["bin1"], d["bin2"] = lo, hi
        d = d[(d["bin1"] >= x1) & (d["bin1"] <= x2) & (d["bin2"] - d["bin1"] <= MAX_DIST)
              & np.isin(d["bin1"], keep) & np.isin(d["bin2"], keep)]
        out.append(d.drop_duplicates(["bin1", "bin2"]))
    df = pd.concat(out, ignore_index=True).rename(columns={"v": "observed"})
    df["oe"] = df["observed"] / expected[(df["bin2"] - df["bin1"]) // HIC_RES]
    df = df[np.isfinite(df["observed"]) & np.isfinite(df["oe"])]
    df[["bin1", "bin2"]] = df[["bin1", "bin2"]].astype(np.int32)
    df[["observed", "oe"]] = df[["observed", "oe"]].astype(np.float32)
    return df

todo = [c for c in CHROMS if not (HIC_DIR / f"{c}.parquet").exists()]
todo.sort(key=lambda c: -CHROM_LEN[c])   # longest first for better load balance
print(f"{len(CHROMS) - len(todo)} chromosomes cached, {len(todo)} to fetch")
t0 = time.time()
with ProcessPoolExecutor(HIC_WORKERS, mp_context=mp.get_context("fork")) as ex:
    futs = {ex.submit(fetch_hic_chrom, c): c for c in todo}
    for fut in tqdm(as_completed(futs), total=len(futs), desc="Hi-C chromosomes"):
        c = futs[fut]
        fut.result().to_parquet(HIC_DIR / f"{c}.parquet", index=False)
print(f"done in {(time.time() - t0) / 60:.1f} min")

In [ ]:
contacts = pd.concat([pd.read_parquet(HIC_DIR / f"{c}.parquet").assign(chrom=c) for c in CHROMS],
                     ignore_index=True)
contacts["dist"] = contacts["bin2"] - contacts["bin1"]
print(f"{len(contacts):,} bin-pair contacts across {contacts['chrom'].nunique()} chromosomes "
      f"({contacts.memory_usage(deep=True).sum() / 1e9:.2f} GB in memory)")
contacts.head()

### 1.6.1 Summary statistics: Hi-C contact vs. distance

Three questions:

1. **How fast does contact decay with distance?** ABC models $O \propto d^{-\gamma}$ with $\gamma \approx 1.02$
   averaged over cell types. We fit $\gamma$ on our node bins. Our node bins are biased toward active
   chromatin, so expect a somewhat shallower slope.
2. **How is O/E distributed, and how many Hi-C edges survive a threshold $\tau$?** This sets the density of
   $E_{\mathrm{HiC}}$ and the exchange rate with $\beta$.
3. **Does Hi-C separate positives from negatives** in the benchmark beyond what distance already does?

In [ ]:
# Distance-decay: mean observed contact in log-spaced distance bins (excluding the diagonal).
off = contacts[contacts["dist"] > 0]
dbins = np.unique(np.round(np.logspace(np.log10(HIC_RES), np.log10(MAX_DIST), 40) / HIC_RES) * HIC_RES)
off_bin = pd.cut(off["dist"], dbins, include_lowest=True)
decay = off.groupby(off_bin, observed=True).agg(d=("dist", "mean"), obs=("observed", "mean"),
                                               oe_med=("oe", "median"), n=("oe", "size"))
fit_mask = (decay["d"] >= 10_000) & (decay["d"] <= 1_000_000)
slope, intercept = np.polyfit(np.log10(decay.loc[fit_mask, "d"]), np.log10(decay.loc[fit_mask, "obs"]), 1)
gamma = -slope
print(f"power-law fit on 10 kb–1 Mb: contact ~ d^-{gamma:.3f}  (ABC average gamma = 1.024)")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
ax = axes[0]
ax.loglog(decay["d"] / 1e3, decay["obs"], marker="o", ms=4, color=C_NEG, label="mean observed (SCALE)")
xx = np.array([10_000, 1_000_000])
ax.loglog(xx / 1e3, 10 ** intercept * xx ** slope, ls="--", lw=1.2, color="#52514e",
          label=rf"fit: $\gamma$ = {gamma:.2f}")
ax.set_xlabel("bin distance (kb)"); ax.set_ylabel("contact"); ax.legend(fontsize=8)
ax.set_title("Distance decay", loc="left")

ax = axes[1]
taus = np.array([1.0, 1.5, 2.0, 3.0, 5.0])
lo = np.log10(off["oe"].clip(lower=1e-2))
ax.hist(lo, bins=80, color=C_NEG)
for t in taus[1:4]:
    ax.axvline(np.log10(t), color="#52514e", lw=0.8, ls=":")
ax.set_yscale("log"); ax.set_xlabel(r"$\log_{10}(O/E)$"); ax.set_ylabel("bin pairs")
ax.set_title("O/E distribution (dotted: τ = 1.5, 2, 3)", loc="left")

ax = axes[2]
for t, color in zip([1.5, 2.0, 3.0], [C_NEG, C_POS, C_3]):
    frac = off.assign(hit=off["oe"] >= t).groupby(off_bin, observed=True)["hit"].mean()
    ax.semilogx(decay["d"] / 1e3, frac.values, color=color, label=f"τ = {t}")
ax.set_xlabel("bin distance (kb)"); ax.set_ylabel(r"fraction with $O/E \geq \tau$"); ax.legend(fontsize=8)
ax.set_title("Where would Hi-C edges land?", loc="left")
fig.tight_layout(); fig.savefig(FIG_DIR / "hic_decay.png", bbox_inches="tight"); plt.show()

In [ ]:
# Edge-budget table: how many Hi-C edges per node would each tau give?
# A bin pair (b1, b2) induces n(b1) * n(b2) node pairs, where n(b) is the number of node centers in bin b.
nodes_per_bin = nodes.groupby(["chrom", "hic_bin"]).size().rename("n")
c2 = (off.merge(nodes_per_bin.rename("n1"), left_on=["chrom", "bin1"], right_index=True)
         .merge(nodes_per_bin.rename("n2"), left_on=["chrom", "bin2"], right_index=True))
c2["node_pairs"] = c2["n1"] * c2["n2"]
rows = []
for t in taus:
    hit = c2[c2["oe"] >= t]
    rows.append({"tau": t, "bin pairs kept": f"{(off['oe'] >= t).mean():.1%}",
                 "Hi-C edges (node pairs)": int(hit["node_pairs"].sum()),
                 "mean Hi-C degree": 2 * hit["node_pairs"].sum() / len(nodes),
                 "median edge distance (kb)": np.median(np.repeat(hit["dist"].values, hit["node_pairs"].values)) / 1e3})
edge_budget = pd.DataFrame(rows).set_index("tau")
print("O/E quantiles (off-diagonal):", off["oe"].quantile([.1, .25, .5, .75, .9, .99]).round(2).to_dict())
edge_budget.round(1)

In [ ]:
# Hi-C at benchmark pairs (bins of element center and TSS).
key = contacts.set_index(["chrom", "bin1", "bin2"])[["observed", "oe"]]
b_e = pairs["elem_center"] // HIC_RES * HIC_RES
b_g = pairs["startTSS"] // HIC_RES * HIC_RES
idx = pd.MultiIndex.from_arrays([pairs["chrom"], np.minimum(b_e, b_g), np.maximum(b_e, b_g)])
pairs[["hic_observed", "hic_oe"]] = key.reindex(idx).to_numpy()
print(f"benchmark pairs with a Hi-C value: {pairs['hic_oe'].notna().mean():.1%}")

# Within distance strata, do positives have higher contact than negatives?
strata = pairs[pairs["dist_kb"] >= 5].groupby(["dist_bin", "label"], observed=True)[["hic_observed", "hic_oe"]].median()
strata = strata.unstack("label").rename(columns={0: "neg", 1: "pos"})
strata.columns = [f"{a} median ({b})" for a, b in strata.columns]
display(strata.round(2))

rows = []
for name, s in [("distance (1/d)", -pairs["dist_kb"]),
                ("Hi-C observed", pairs["hic_observed"].fillna(0)),
                ("Hi-C O/E", pairs["hic_oe"].fillna(0))]:
    rows.append({"score": name, "pooled AUPRC": average_precision_score(pairs["label"], s),
                 "precision @70% recall": precision_at_recall(pairs["label"], s)})
pd.DataFrame(rows).set_index("score").round(3)

**Reading the table.** O/E removes the distance trend by construction, so its AUPRC is close to the base rate.
Raw observed contact still contains distance, which is why ABC uses it. The useful signal for our model is
that positives sit at higher O/E than negatives within the same distance stratum: those are the shortcut
edges a walk can take.

## 1.7 Node sequences (GRCh38)

`pyfaidx` indexes `hg38.fa` on first use (`.fai`, about 30 s) and then gives random access. We extract each
node's sequence (uppercased; UCSC soft-masks repeats in lowercase) and run sanity checks: GC content (open
chromatin and promoters are GC-rich) and N content (should be about zero after the blocklist filter).

HyenaDNA embeddings $s_v$ are computed in Part 2. The tiny 1k-context checkpoint covers about 97% of nodes as
is, and longer merged nodes are centered and cropped.

In [ ]:
fasta = pyfaidx.Fasta(str(PATHS["hg38"]), as_raw=True, sequence_always_upper=True)
seqs = [fasta[c][s:e] for c, s, e in tqdm(zip(nodes["chrom"], nodes["start"], nodes["end"]),
                                          total=len(nodes), desc="sequences", mininterval=2)]
node_seq = pd.DataFrame({"node_id": nodes["node_id"], "sequence": seqs})
lens = node_seq["sequence"].str.len()
assert (lens.values == nodes["length"].values).all()
gc = node_seq["sequence"].str.count("[GC]") / lens
nodes["gc"] = gc.values
nodes["n_frac"] = (node_seq["sequence"].str.count("N") / lens).values
print(f"GC content: promoters {nodes.loc[nodes.is_promoter, 'gc'].mean():.3f}, "
      f"ATAC-only {nodes.loc[~nodes.is_promoter, 'gc'].mean():.3f} (genome average ~0.41)")
print(f"nodes containing any N: {(nodes['n_frac'] > 0).sum()} | nodes <= 1 kb: {(nodes['length'] <= 1000).mean():.1%}")

# Spot check: a known element sequence from the benchmark matches the reference.
r = pairs.iloc[0]
print(r["name"], fasta[r["chrom"]][int(r["chromStart"]):int(r["chromStart"]) + 60])

## 1.8 Save processed tables

In [ ]:
pair_cols = ["pair_id", "Dataset", "chrom", "chromStart", "chromEnd", "elem_center", "elem_len",
             "measuredGeneSymbol", "measuredGeneEnsemblId", "startTSS", "distanceToTSS", "dist_kb",
             "label", "EffectSize", "pValueAdjusted", "elem_node", "elem_overlap_bp", "n_elem_nodes",
             "elem_in_graph", "gene_node", "hic_observed", "hic_oe"]
pairs[pair_cols].to_parquet(PROC / "crispr_pairs.parquet", index=False)
nodes.to_parquet(PROC / "nodes.parquet", index=False)
node_seq.to_parquet(PROC / "node_sequences.parquet", index=False)
genes.to_parquet(PROC / "genes.parquet", index=False)

for f in sorted(PROC.glob("*.parquet")):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:8.1f} MB")
print(f"{'hic/*.parquet':28s} {sum(f.stat().st_size for f in HIC_DIR.glob('*.parquet')) / 1e6:8.1f} MB")

## Part 1 summary

What we have now, in one coordinate system:

| Object | Size | Key columns |
|---|---|---|
| `nodes` | ~150–170k candidate elements | `node_id, chrom, start, end, log_atac, is_promoter, promoter_genes, hic_bin, gc` |
| `contacts` | bin pairs within 5 Mb | `chrom, bin1, bin2, observed, oe` |
| `pairs` | 10,356 benchmark pairs | `elem_node, gene_node, label, Dataset, dist_kb, hic_observed, hic_oe` |
| `node_seq` | one sequence per node | `node_id, sequence` |

**Next (Part 2: graph construction).** For each chromosome: $E_{\mathrm{lin}}$ links consecutive `node_id`s, and
$E_{\mathrm{HiC}}$ joins node pairs whose bins have $O/E \geq \tau$ (pick $\tau$ from the edge-budget table).
Then build PyG `Data` objects with node features $[a_v; s_v]$ and edge attributes (type, signed distance, O/E).